In [ ]:
# %% CELL 1 — Setup & Installs
# ------------------------------------------------------------------------------
get_ipython().system('pip install biopython transformers torch scikit-learn xgboost matplotlib scipy -q')

import os, pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, roc_auc_score, accuracy_score, roc_curve)
import xgboost as xgb

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Mounted at /content/drive
Using device: cuda


In [ ]:
# %% CELL 2 — Paths
# ------------------------------------------------------------------------------
BASE = "/content/drive/MyDrive/AI/public projects/"
CKPT = BASE + "checkpoints/"

pos_path            = BASE + "VF_positive_subset_final_2.faa"
neg_path            = BASE + "VF_negative_subset_final.faa"
predicted_orfs_path = BASE + "predicted_orfs.faa"
blast_path          = BASE + "orfs_vs_vfdb_full.tsv"        # genome ORFs vs VFDB
pfam_path           = BASE + "pfam_hits.tbl"                # genome ORFs vs Pfam
train_blast_path    = BASE + "train_vs_vfdb.tsv"            # training seqs vs VFDB (Model 3 dependency)

checkpoint_path      = CKPT + "vf_pipeline_checkpoint.pth"
orf_embed_path       = CKPT + "orf_embeddings.pkl"
train_dl_embed_path  = CKPT + "train_dl_embeddings.pkl"
train_pfam_path      = CKPT + "train_pfam_counts.pkl"
train_cdd_tblout     = CKPT + "train_cdd_hits.tlout"         # note: this is the misspelled filename you listed

meta_model_path      = CKPT + "meta_model.json"              # current primary = Model 3
meta_model_v2_path   = CKPT + "meta_model_v2_cdd_only.json"
meta_model_v3_path   = CKPT + "meta_model_v3_with_blast.json"

all_paths = [pos_path, neg_path, predicted_orfs_path, blast_path, pfam_path, train_blast_path,
             checkpoint_path, orf_embed_path, train_dl_embed_path, train_pfam_path,
             meta_model_path, meta_model_v2_path, meta_model_v3_path]

for p in all_paths:
    print(f"{'OK' if os.path.exists(p) else 'MISSING'}: {p}")

OK: /content/drive/MyDrive/AI/public projects/VF_positive_subset_final_2.faa
OK: /content/drive/MyDrive/AI/public projects/VF_negative_subset_final.faa
OK: /content/drive/MyDrive/AI/public projects/predicted_orfs.faa
OK: /content/drive/MyDrive/AI/public projects/orfs_vs_vfdb_full.tsv
OK: /content/drive/MyDrive/AI/public projects/pfam_hits.tbl
OK: /content/drive/MyDrive/AI/public projects/train_vs_vfdb.tsv
OK: /content/drive/MyDrive/AI/public projects/checkpoints/vf_pipeline_checkpoint.pth
OK: /content/drive/MyDrive/AI/public projects/checkpoints/orf_embeddings.pkl
OK: /content/drive/MyDrive/AI/public projects/checkpoints/train_dl_embeddings.pkl
OK: /content/drive/MyDrive/AI/public projects/checkpoints/train_pfam_counts.pkl
OK: /content/drive/MyDrive/AI/public projects/checkpoints/meta_model.json
OK: /content/drive/MyDrive/AI/public projects/checkpoints/meta_model_v2_cdd_only.json
OK: /content/drive/MyDrive/AI/public projects/checkpoints/meta_model_v3_with_blast.json


In [ ]:
# %% CELL 3 — Model Architecture (needed for attention weights / entropy analysis)
# ------------------------------------------------------------------------------
model_name = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, x, mask):
        scores = self.attention(x).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        context_vector = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return context_vector, weights

class GeneEncoder(nn.Module):
    def __init__(self, transformer_model_name, unfreeze_last_n_layers=2):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_model_name)
        self.hidden_dim = self.transformer.config.hidden_size
        self.n_layers = self.transformer.config.num_hidden_layers
        for param in self.transformer.parameters():
            param.requires_grad = False
        for name, param in self.transformer.named_parameters():
            for i in range(self.n_layers - unfreeze_last_n_layers, self.n_layers):
                if f"layer.{i}." in name:
                    param.requires_grad = True
        self.layer_weight_logits = nn.Parameter(torch.zeros(self.n_layers + 1))
        self.attn_pool = AttentionPooling(self.hidden_dim)
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        all_layers = torch.stack(outputs.hidden_states, dim=0)
        layer_weights = F.softmax(self.layer_weight_logits, dim=0)
        combined = (all_layers * layer_weights.view(-1, 1, 1, 1)).sum(0)
        context_vector, attn_weights = self.attn_pool(combined, attention_mask)
        return context_vector, attn_weights, layer_weights

class VFClassifierHead(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

class GenomicWindowEncoder(nn.Module):
    """Kept for checkpoint-loading compatibility only — NOT used."""
    def __init__(self, gene_embed_dim, n_heads=4, n_layers=1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=gene_embed_dim, nhead=n_heads, dim_feedforward=gene_embed_dim * 2,
            dropout=0.1, batch_first=True
        )
        self.context_transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.window_pool = AttentionPooling(gene_embed_dim)
        self.window_classifier = nn.Sequential(
            nn.Linear(gene_embed_dim, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1)
        )
    def forward(self, gene_embeddings, gene_mask):
        src_key_padding_mask = (gene_mask == 0)
        contextualized = self.context_transformer(gene_embeddings, src_key_padding_mask=src_key_padding_mask)
        window_embedding, gene_importance = self.window_pool(contextualized, gene_mask)
        window_logits = self.window_classifier(window_embedding)
        return window_embedding, gene_importance, window_logits

class FullVFPipeline(nn.Module):
    def __init__(self, transformer_model_name, unfreeze_last_n_layers=2):
        super().__init__()
        self.gene_encoder = GeneEncoder(transformer_model_name, unfreeze_last_n_layers)
        self.classifier = VFClassifierHead(self.gene_encoder.hidden_dim)
        self.window_encoder = GenomicWindowEncoder(self.gene_encoder.hidden_dim)  # unused
    def encode_gene(self, input_ids, attention_mask):
        context_vector, attn_weights, layer_weights = self.gene_encoder(input_ids, attention_mask)
        logits = self.classifier(context_vector)
        return logits, context_vector, attn_weights

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
# %% CELL 4 — Load Trained Stage 1 Checkpoint (NO retraining)
# ------------------------------------------------------------------------------
model = FullVFPipeline(model_name, unfreeze_last_n_layers=2).to(device)
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded Stage 1 model from epoch {ckpt['epoch']+1}, val AUC {ckpt['best_val_auc']:.4f}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  136MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded Stage 1 model from epoch 6, val AUC 0.9569


In [ ]:
# %% CELL 5 — Load Training Sequences + Genome ORFs
# ------------------------------------------------------------------------------
sequences, labels = [], []
for record in SeqIO.parse(pos_path, "fasta"):
    sequences.append(str(record.seq)); labels.append(1)
for record in SeqIO.parse(neg_path, "fasta"):
    sequences.append(str(record.seq)); labels.append(0)
print(f"Training sequences: {len(sequences)} | Positive: {sum(labels)} Negative: {len(labels)-sum(labels)}")

orf_seqs = {}
for record in SeqIO.parse(predicted_orfs_path, "fasta"):
    orf_seqs[record.id] = str(record.seq).rstrip("*")
print(f"Genome ORF sequences: {len(orf_seqs)}")

y = np.array(labels)

Training sequences: 22207 | Positive: 11584 Negative: 10623
Genome ORF sequences: 34074


In [ ]:
# %% CELL 6 — Load Precomputed Embeddings + Domain Counts (no recompute)
# ------------------------------------------------------------------------------
with open(orf_embed_path, "rb") as f:
    orf_embeddings = pickle.load(f)
with open(train_dl_embed_path, "rb") as f:
    dl_embeddings_matrix = pickle.load(f)
with open(train_pfam_path, "rb") as f:
    saved = pickle.load(f)
    train_pfam_counts = saved["counts"]

print(f"Loaded {len(orf_embeddings)} ORF embeddings, {dl_embeddings_matrix.shape} train embeddings")
print(f"Loaded CDD domain counts for {len(train_pfam_counts)} training genes")

Loaded 34074 ORF embeddings, (22207, 480) train embeddings
Loaded CDD domain counts for 17234 training genes


In [ ]:
# %% CELL 7 — Load all three Meta-Models
# ------------------------------------------------------------------------------
meta_model = xgb.XGBClassifier()       # primary = Model 3 (CDD + de-leaked BLAST)
meta_model.load_model(meta_model_path)

model2_reloaded = xgb.XGBClassifier()  # archived Model 2 (CDD only)
model2_reloaded.load_model(meta_model_v2_path)

model3 = xgb.XGBClassifier()           # explicit Model 3 copy (identical to primary)
model3.load_model(meta_model_v3_path)

print("All three meta-models loaded: meta_model (primary=v3), model2_reloaded (v2), model3 (v3 explicit)")

All three meta-models loaded: meta_model (primary=v3), model2_reloaded (v2), model3 (v3 explicit)


In [ ]:
# %% CELL 8 — Load BLAST (genome + training) and Pfam Results
# ------------------------------------------------------------------------------
blast_cols = ["query_id", "subject_id", "pident", "length", "mismatch",
              "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore", "extra_col"]
blast_hits = pd.read_csv(blast_path, sep="\t", names=blast_cols, header=None)
best_hits = blast_hits.sort_values("evalue").drop_duplicates("query_id", keep="first")
best_evalue_lookup = dict(zip(best_hits["query_id"], best_hits["evalue"]))
best_hits_bitscore_lookup = dict(zip(best_hits["query_id"], best_hits["bitscore"]))
blast_overlap = set(orf_seqs.keys()).intersection(set(blast_hits['query_id']))
print(f"Genome BLAST: {len(blast_hits)} hits, {len(blast_overlap)} ORFs with a VFDB hit")

def parse_hmmer_tblout(tblout_path, evalue_thresh=1e-5):
    domain_counts, domain_hits = {}, {}
    if not os.path.exists(tblout_path) or os.path.getsize(tblout_path) == 0:
        return domain_counts, domain_hits
    with open(tblout_path) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.split()
            if len(parts) < 13: continue
            gene_id = parts[3]
            try: domain_evalue = float(parts[6])
            except ValueError: continue
            if domain_evalue <= evalue_thresh:
                domain_counts[gene_id] = domain_counts.get(gene_id, 0) + 1
    return domain_counts, domain_hits

genome_pfam_counts, _ = parse_hmmer_tblout(pfam_path)
print(f"Genome ORFs with >=1 Pfam domain: {len(genome_pfam_counts)}")

# Training BLAST (needed for Model 3 feature reconstruction) — only if the file exists
if os.path.exists(train_blast_path):
    import re
    def extract_vfg_id(full_id):
        match = re.match(r"(VFG\d+)", full_id)
        return match.group(1) if match else full_id

    train_own_vfg = {}
    idx = 0
    for rec in SeqIO.parse(pos_path, "fasta"):
        train_own_vfg[f"train_{idx}"] = extract_vfg_id(rec.id)
        idx += 1
    for rec in SeqIO.parse(neg_path, "fasta"):
        idx += 1

    train_blast_cols = ["query_id", "subject_id", "pident", "length", "mismatch",
                         "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]
    train_blast_hits = pd.read_csv(train_blast_path, sep="\t", names=train_blast_cols, header=None)
    train_blast_hits["subject_vfg"] = train_blast_hits["subject_id"].apply(extract_vfg_id)
    train_blast_hits["query_own_vfg"] = train_blast_hits["query_id"].map(train_own_vfg)
    is_self = (train_blast_hits["subject_vfg"] == train_blast_hits["query_own_vfg"]) & train_blast_hits["query_own_vfg"].notna()
    train_blast_hits_filtered = train_blast_hits[~is_self].copy()
    train_best_hits = train_blast_hits_filtered.sort_values("evalue").drop_duplicates("query_id", keep="first")
    train_bitscore_lookup = dict(zip(train_best_hits["query_id"], train_best_hits["bitscore"]))
    print(f"Training BLAST (de-leaked): {len(train_bitscore_lookup)} training genes with a non-self hit")
else:
    print("WARNING: train_vs_vfdb.tsv missing — Model 3 training-feature reconstruction unavailable this session")
    train_bitscore_lookup = {}

Genome BLAST: 11516 hits, 3611 ORFs with a VFDB hit
Genome ORFs with >=1 Pfam domain: 625
Training BLAST (de-leaked): 11277 training genes with a non-self hit


In [ ]:
# %% CELL 9 — Rebuild results_final using Model 3 (primary) scores on genome ORFs
# ------------------------------------------------------------------------------
genome_gene_ids = list(orf_seqs.keys())
genome_X_final = []
for gid in genome_gene_ids:
    dl_embedding = orf_embeddings[gid].numpy()
    pfam_count = genome_pfam_counts.get(gid, 0)
    blast_bitscore = best_hits_bitscore_lookup.get(gid, 0.0)
    genome_X_final.append(list(dl_embedding) + [pfam_count, blast_bitscore])
genome_X_final = np.array(genome_X_final)

genome_vf_proba_final = meta_model.predict_proba(genome_X_final)[:, 1]

results_final = pd.DataFrame({
    "gene_id": genome_gene_ids,
    "meta_model_vf_probability": genome_vf_proba_final,
    "has_blast_hit": [gid in blast_overlap for gid in genome_gene_ids],
    "blast_evalue": [best_evalue_lookup.get(gid, np.nan) for gid in genome_gene_ids],
    "blast_bitscore": [best_hits_bitscore_lookup.get(gid, np.nan) for gid in genome_gene_ids],
    "pfam_domain_count": [genome_pfam_counts.get(gid, 0) for gid in genome_gene_ids],
})
results_final["evidence_tier"] = "no_evidence"
results_final.loc[(results_final["has_blast_hit"]) | (results_final["pfam_domain_count"] > 0), "evidence_tier"] = "corroborated"
results_final.loc[(results_final["has_blast_hit"]) & (results_final["pfam_domain_count"] > 0), "evidence_tier"] = "strongly_corroborated"
results_final = results_final.sort_values("meta_model_vf_probability", ascending=False)

print(f"Total genome ORFs scored: {len(results_final)}")
print(f"High-confidence VF genes (proba > 0.8): {(results_final['meta_model_vf_probability'] > 0.8).sum()}")
print(results_final["evidence_tier"].value_counts())

Total genome ORFs scored: 34074
High-confidence VF genes (proba > 0.8): 7060
evidence_tier
no_evidence              29917
corroborated              4078
strongly_corroborated       79
Name: count, dtype: int64


In [ ]:
# %% CELL V1 — Genome selection for validation
# ------------------------------------------------------------------------------
# Pick 2-3 genomes from different species than the original test_sample.fna
# RefSeq assembly accessions (complete genomes, moderate size, well-annotated pathogens):
validation_genomes = {
    "ecoli_O157H7":  "GCF_000008865.2",  # E. coli O157:H7 Sakai — classic pathogen, VFDB-rich
    "salmonella":    "GCF_000006945.2",  # Salmonella enterica Typhimurium LT2
    "pseudomonas":   "GCF_000006765.1",  # Pseudomonas aeruginosa PAO1
}
print("Selected validation genomes:", list(validation_genomes.keys()))

Selected validation genomes: ['ecoli_O157H7', 'salmonella', 'pseudomonas']


In [ ]:
# %% CELL V2 — Download genome FASTA files from NCBI
# ------------------------------------------------------------------------------
import subprocess

os.makedirs("/content/validation_genomes", exist_ok=True)

def download_genome(accession, name):
    """Download a RefSeq genome assembly FASTA via NCBI datasets tool."""
    out_dir = f"/content/validation_genomes/{name}"
    os.makedirs(out_dir, exist_ok=True)
    # Use NCBI datasets CLI
    result = subprocess.run(
        ["datasets", "download", "genome", "accession", accession,
         "--filename", f"{out_dir}/{name}.zip"],
        capture_output=True, text=True
    )
    print(f"{name}: {result.returncode}, {result.stderr[:300]}")
    return result.returncode == 0

# Install NCBI datasets CLI if not present
subprocess.run(["curl", "-o", "datasets",
                 "https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets"],
                capture_output=True)
subprocess.run(["chmod", "+x", "datasets"])
os.environ["PATH"] = "/content:" + os.environ["PATH"]

for name, accession in validation_genomes.items():
    download_genome(accession, name)

ecoli_O157H7: 0, Collecting 1 genome record [------------------------------------------------]   0% 0/1
salmonella: 0, Collecting 1 genome record [------------------------------------------------]   0% 0/1
pseudomonas: 0, Collecting 1 genome record [------------------------------------------------]   0% 0/1


In [ ]:
# %% CELL V3 — Unzip and extract genome FASTA files
# ------------------------------------------------------------------------------
import zipfile, glob

genome_fasta_paths = {}
for name in validation_genomes:
    zip_path = f"/content/validation_genomes/{name}/{name}.zip"
    extract_dir = f"/content/validation_genomes/{name}/extracted"
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_dir)
        fna_files = glob.glob(f"{extract_dir}/**/*.fna", recursive=True)
        if fna_files:
            genome_fasta_paths[name] = fna_files[0]
            print(f"{name}: {fna_files[0]}")
        else:
            print(f"{name}: no .fna found after extraction")
    else:
        print(f"{name}: zip missing, download may have failed")

ecoli_O157H7: /content/validation_genomes/ecoli_O157H7/extracted/ncbi_dataset/data/GCF_000008865.2/GCF_000008865.2_ASM886v2_genomic.fna
salmonella: /content/validation_genomes/salmonella/extracted/ncbi_dataset/data/GCF_000006945.2/GCF_000006945.2_ASM694v2_genomic.fna
pseudomonas: /content/validation_genomes/pseudomonas/extracted/ncbi_dataset/data/GCF_000006765.1/GCF_000006765.1_ASM676v1_genomic.fna


In [ ]:
# %% CELL V4 — Reusable functions, matching the exact main-genome pipeline
# ------------------------------------------------------------------------------
from Bio.Seq import Seq

def pad_sequence(seq):
    remainder = len(seq) % 3
    if remainder != 0:
        seq += 'N' * (3 - remainder)
    return seq

def find_orfs(seq, min_len=100):
    """Same 6-frame ORF finder used to generate predicted_orfs.faa originally."""
    orfs = []
    for strand, nuc in [(+1, seq), (-1, seq.reverse_complement())]:
        for frame in range(3):
            trans = nuc[frame:].translate(to_stop=False)
            aa_seq = str(trans)
            start = 0
            while True:
                idx = aa_seq.find('M', start)
                if idx == -1: break
                stop_idx = aa_seq.find('*', idx)
                if stop_idx == -1: break
                if stop_idx - idx >= min_len:
                    orfs.append(aa_seq[idx:stop_idx])
                start = idx + 1
    return orfs

def call_orfs_for_genome(fasta_path, min_len=100):
    contigs = list(SeqIO.parse(fasta_path, "fasta"))
    all_orfs = []
    for contig in contigs:
        padded = pad_sequence(contig.seq)
        all_orfs.extend(find_orfs(padded, min_len=min_len))
    orf_records = {f"ORF_{i+1}": orf for i, orf in enumerate(all_orfs)}
    print(f"  {len(contigs)} contigs -> {len(orf_records)} ORFs")
    return orf_records

def get_batched_embeddings(seqs_dict, model, tokenizer, device, batch_size=16, max_length=512):
    """Batched Stage 1 embedding generation for a dict of {id: sequence}."""
    ids = list(seqs_dict.keys())
    embeddings = {}
    model.gene_encoder.eval()
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i+batch_size]
        batch_seqs = [seqs_dict[gid] for gid in batch_ids]
        enc = tokenizer(batch_seqs, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        with torch.no_grad():
            context_vectors, _, _ = model.gene_encoder(
                enc["input_ids"].to(device), enc["attention_mask"].to(device)
            )
        for gid, vec in zip(batch_ids, context_vectors.cpu()):
            embeddings[gid] = vec
        if i % (batch_size * 20) == 0:
            print(f"    embedded {i}/{len(ids)}")
    return embeddings

In [ ]:
# %% Install NCBI BLAST+ (not persisted across Colab sessions)
# ------------------------------------------------------------------------------
!apt-get install ncbi-blast+ -y -qq
!which blastp

Selecting previously unselected package ncbi-data.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb ...
Unpacking ncbi-blast+ (2.12.0+ds-3build1) ...
Setting up ncbi-data (6.1.20170106+dfsg1-9) ...
Setting up ncbi-blast+ (2.12.0+ds-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...
/usr/bin/blastp


In [ ]:
# %% CELL V5 — Full pipeline per validation genome
# ------------------------------------------------------------------------------
def run_full_pipeline(name, fasta_path, model, tokenizer, device, meta_model, vfdb_db="/content/vfdb/vfdb_db"):
    print(f"\n=== {name} ===")
    out_dir = f"/content/validation_genomes/{name}"

    # 1. ORF calling
    print("Calling ORFs...")
    orf_dict = call_orfs_for_genome(fasta_path)

    orf_faa_path = f"{out_dir}/predicted_orfs.faa"
    with open(orf_faa_path, "w") as f:
        for gid, seq in orf_dict.items():
            f.write(f">{gid}\n{seq}\n")

    # 2. Stage 1 embeddings (batched)
    print("Generating embeddings (this is the slow step)...")
    embeddings = get_batched_embeddings(orf_dict, model, tokenizer, device)

    # 3. BLAST vs VFDB
    print("Running BLAST vs VFDB...")
    blast_out = f"{out_dir}/orfs_vs_vfdb.tsv"
    result = subprocess.run(
        ["blastp", "-query", orf_faa_path, "-db", vfdb_db, "-out", blast_out,
         "-evalue", "1e-5", "-num_threads", "2",
         "-outfmt", "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"],
        capture_output=True
    )
    print(f"  BLAST return code: {result.returncode}, output size: {os.path.getsize(blast_out) if os.path.exists(blast_out) else 0}")

    blast_cols = ["query_id", "subject_id", "pident", "length", "mismatch",
                  "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]
    if os.path.getsize(blast_out) > 0:
        blast_df = pd.read_csv(blast_out, sep="\t", names=blast_cols, header=None)
        best = blast_df.sort_values("evalue").drop_duplicates("query_id", keep="first")
        bitscore_lookup = dict(zip(best["query_id"], best["bitscore"]))
        evalue_lookup = dict(zip(best["query_id"], best["evalue"]))
        blast_hit_ids = set(best["query_id"])
    else:
        bitscore_lookup, evalue_lookup, blast_hit_ids = {}, {}, set()

    # 4. Score with Model 3 (DL embedding + CDD count [0, no CDD scan for validation genomes] + BLAST bitscore)
    print("Scoring with Model 3...")
    gene_ids = list(orf_dict.keys())
    X_val = []
    for gid in gene_ids:
        emb = embeddings[gid].numpy()
        pfam_count = 0  # skip CDD/Pfam scan for speed — flagged as a simplification below
        bitscore = bitscore_lookup.get(gid, 0.0)
        X_val.append(list(emb) + [pfam_count, bitscore])
    X_val = np.array(X_val)
    proba = meta_model.predict_proba(X_val)[:, 1]

    results_val = pd.DataFrame({
        "gene_id": gene_ids,
        "meta_model_vf_probability": proba,
        "has_blast_hit": [g in blast_hit_ids for g in gene_ids],
        "blast_evalue": [evalue_lookup.get(g, np.nan) for g in gene_ids],
    })
    results_val.to_csv(f"{out_dir}/final_predictions.csv", index=False)
    return results_val

validation_results = {}
for name, fasta_path in genome_fasta_paths.items():
    validation_results[name] = run_full_pipeline(name, fasta_path, model, tokenizer, device, meta_model)


=== ecoli_O157H7 ===
Calling ORFs...


/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


  3 contigs -> 36138 ORFs
Generating embeddings (this is the slow step)...
    embedded 0/36138
    embedded 320/36138
    embedded 640/36138
    embedded 960/36138
    embedded 1280/36138
    embedded 1600/36138
    embedded 1920/36138
    embedded 2240/36138
    embedded 2560/36138
    embedded 2880/36138
    embedded 3200/36138
    embedded 3520/36138
    embedded 3840/36138
    embedded 4160/36138
    embedded 4480/36138
    embedded 4800/36138
    embedded 5120/36138
    embedded 5440/36138
    embedded 5760/36138
    embedded 6080/36138
    embedded 6400/36138
    embedded 6720/36138
    embedded 7040/36138
    embedded 7360/36138
    embedded 7680/36138
    embedded 8000/36138
    embedded 8320/36138
    embedded 8640/36138
    embedded 8960/36138
    embedded 9280/36138
    embedded 9600/36138
    embedded 9920/36138
    embedded 10240/36138
    embedded 10560/36138
    embedded 10880/36138
    embedded 11200/36138
    embedded 11520/36138
    embedded 11840/36138
    embedded 

/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


  2 contigs -> 32940 ORFs
Generating embeddings (this is the slow step)...
    embedded 0/32940
    embedded 320/32940
    embedded 640/32940
    embedded 960/32940
    embedded 1280/32940
    embedded 1600/32940
    embedded 1920/32940
    embedded 2240/32940
    embedded 2560/32940
    embedded 2880/32940
    embedded 3200/32940
    embedded 3520/32940
    embedded 3840/32940
    embedded 4160/32940
    embedded 4480/32940
    embedded 4800/32940
    embedded 5120/32940
    embedded 5440/32940
    embedded 5760/32940
    embedded 6080/32940
    embedded 6400/32940
    embedded 6720/32940
    embedded 7040/32940
    embedded 7360/32940
    embedded 7680/32940
    embedded 8000/32940
    embedded 8320/32940
    embedded 8640/32940
    embedded 8960/32940
    embedded 9280/32940
    embedded 9600/32940
    embedded 9920/32940
    embedded 10240/32940
    embedded 10560/32940
    embedded 10880/32940
    embedded 11200/32940
    embedded 11520/32940
    embedded 11840/32940
    embedded 

/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


  1 contigs -> 50146 ORFs
Generating embeddings (this is the slow step)...
    embedded 0/50146
    embedded 320/50146
    embedded 640/50146
    embedded 960/50146
    embedded 1280/50146
    embedded 1600/50146
    embedded 1920/50146
    embedded 2240/50146
    embedded 2560/50146
    embedded 2880/50146
    embedded 3200/50146
    embedded 3520/50146
    embedded 3840/50146
    embedded 4160/50146
    embedded 4480/50146
    embedded 4800/50146
    embedded 5120/50146
    embedded 5440/50146
    embedded 5760/50146
    embedded 6080/50146
    embedded 6400/50146
    embedded 6720/50146
    embedded 7040/50146
    embedded 7360/50146
    embedded 7680/50146
    embedded 8000/50146
    embedded 8320/50146
    embedded 8640/50146
    embedded 8960/50146
    embedded 9280/50146
    embedded 9600/50146
    embedded 9920/50146
    embedded 10240/50146
    embedded 10560/50146
    embedded 10880/50146
    embedded 11200/50146
    embedded 11520/50146
    embedded 11840/50146
    embedded 

In [ ]:
import pandas as pd
import os

# Ensure your BASE path is defined (from your earlier cells)
BASE = "/content/drive/MyDrive/AI/public projects/"
validation_dir = os.path.join(BASE, "validation_results/")
os.makedirs(validation_dir, exist_ok=True)

# 1. Save individual genome results to Drive and add genome name
for name, df in validation_results.items():
    df['genome_name'] = name  # Add identifier column

    # Save individual CSV to Drive
    drive_path = os.path.join(validation_dir, f"validation_{name}_predictions.csv")
    df.to_csv(drive_path, index=False)
    print(f"Saved {name} to Drive: {drive_path}")

# 2. Aggregate into a Master DataFrame
master_validation_df = pd.concat(validation_results.values(), ignore_index=True)
master_path = os.path.join(BASE, "master_validation_predictions.csv")
master_validation_df.to_csv(master_path, index=False)
print(f"\n✅ Master validation file saved to: {master_path}")

# 3. Generate and print a Validation Summary Table
print("\n" + "="*60)
print("VALIDATION SUMMARY METRICS")
print("="*60)

summary_stats = []
for name, df in validation_results.items():
    total_orfs = len(df)
    high_conf = (df['meta_model_vf_probability'] > 0.8).sum()
    blast_hits = df['has_blast_hit'].sum()

    summary_stats.append({
        'Genome': name,
        'Total ORFs': total_orfs,
        'High-Confidence VFs (>0.8)': high_conf,
        'BLAST Hits (VFDB)': blast_hits,
        '% High-Confidence': f"{(high_conf / total_orfs) * 100:.2f}%"
    })

summary_df = pd.DataFrame(summary_stats)
print(summary_df.to_string(index=False))

# Save summary to Drive
summary_path = os.path.join(BASE, "validation_summary_metrics.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\n✅ Summary metrics saved to: {summary_path}")

Saved ecoli_O157H7 to Drive: /content/drive/MyDrive/AI/public projects/validation_results/validation_ecoli_O157H7_predictions.csv
Saved salmonella to Drive: /content/drive/MyDrive/AI/public projects/validation_results/validation_salmonella_predictions.csv
Saved pseudomonas to Drive: /content/drive/MyDrive/AI/public projects/validation_results/validation_pseudomonas_predictions.csv

✅ Master validation file saved to: /content/drive/MyDrive/AI/public projects/master_validation_predictions.csv

VALIDATION SUMMARY METRICS
      Genome  Total ORFs  High-Confidence VFs (>0.8)  BLAST Hits (VFDB) % High-Confidence
ecoli_O157H7       36138                        6430                  0            17.79%
  salmonella       32940                        5243                  0            15.92%
 pseudomonas       50146                       13817                  0            27.55%

✅ Summary metrics saved to: /content/drive/MyDrive/AI/public projects/validation_summary_metrics.csv


In [ ]:
# %% CELL V6 — Compare evidence-tier patterns across all genomes
# ------------------------------------------------------------------------------
print("=== Cross-Genome Generalization Check ===\n")

summary_rows = []
# Original genome (from results_final, already in memory)
orig_high_conf = (results_final["meta_model_vf_probability"] > 0.8).mean() * 100
orig_blast_conf_median = results_final[results_final["has_blast_hit"]]["meta_model_vf_probability"].median()
summary_rows.append({
    "genome": "original (test_sample.fna)",
    "total_orfs": len(results_final),
    "pct_high_confidence": orig_high_conf,
    "blast_confirmed_median_proba": orig_blast_conf_median,
})

for name, res in validation_results.items():
    high_conf_pct = (res["meta_model_vf_probability"] > 0.8).mean() * 100
    bc_median = res[res["has_blast_hit"]]["meta_model_vf_probability"].median() if res["has_blast_hit"].sum() > 0 else np.nan
    summary_rows.append({
        "genome": name,
        "total_orfs": len(res),
        "pct_high_confidence": high_conf_pct,
        "blast_confirmed_median_proba": bc_median,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df)

=== Cross-Genome Generalization Check ===

                       genome  total_orfs  pct_high_confidence  \
0  original (test_sample.fna)       34074            20.719610   
1                ecoli_O157H7       36138            17.792905   
2                  salmonella       32940            15.916818   
3                 pseudomonas       50146            27.553544   

   blast_confirmed_median_proba  
0                      0.350767  
1                           NaN  
2                           NaN  
3                           NaN  
